In [1]:
import pandas as pd
import requests
import time

# MODEL (It is Ollama's default local API endpoint. When Ollama is installed and running on a machine, it exposes its generation 
# API on port 11434, so applications use http://localhost:11434/api/generate to send prompts to the model.)

MODEL = "llama3.2"
OLLAMA_URL = "http://localhost:11434/api/generate"

# price per 1 million token (set according to your deployment)
# local ollama inference has no per-token API charge.

INPUT_PRICE = 0.0
OUTPUT_PRICE = 0.0

# LOAD QUESTIONS
df = pd.read_csv("ai_teacher_20_questions.csv")
print(f"Total question: {len(df)}")

# AI TEACHER PROMPT
instructions = """You are an expert AI Teacher.

Explain the concept using simple layman terms.
Keep the answer precise and easy to understand.
Give the answer in approximately 3 lines. """

# STORAGE
answers = []
latencies = []
input_tokens = []
output_tokens = []
total_tokens = []
input_costs = []
output_costs = []
total_costs = []

# Generate Answer
for index, row in df.iterrows():
    question = str(row["User Query"])

    try:
        prompt = f"{instructions}\n\nQuestion: {question}"

        # Start Timer
        start_time = time.perf_counter()

        # OLLAMA API CALL
        response = requests.post(OLLAMA_URL, json={"model": MODEL, "prompt": prompt, "stream": False}, timeout=300)
        response.raise_for_status()
        result = response.json()
        
        # END TIMER
        latency = time.perf_counter() - start_time

        # Answer
        answer = result["response"].strip()

        # Token usage
        input_token_count = result.get("prompt_eval_count", 0)
        output_token_count = result.get("eval_count", 0)
        total_token_count = (input_token_count + output_token_count)

        # COST (zero for local inference)
        input_cost = (input_token_count / 1_000_000) * INPUT_PRICE
        output_cost = (output_token_count / 1_000_000) * OUTPUT_PRICE

        total_cost = input_cost + output_cost

        # Store Values
        answers.append(answer)
        latencies.append(latency)
        input_tokens.append(input_token_count)
        output_tokens.append(output_token_count)
        total_tokens.append(total_token_count)
        input_costs.append(input_cost)
        output_costs.append(output_cost)
        total_costs.append(total_cost)

    except Exception as e:
        print(f"Error on row {index}: {e}")

        answers.append("ERROR")
        latencies.append(0)
        input_tokens.append(0)
        output_tokens.append(0)
        total_tokens.append(0)
        input_costs.append(0)
        output_costs.append(0)
        total_costs.append(0)        

# ADD RESULTS TO DATAFRAME
df["GeneratedAnswer"] = answers
df["Latency"] = latencies
df["InputTokens"] = input_tokens
df["OutputTokens"] = output_tokens
df["TotalTokens"] = total_tokens
df["InputCost"] = input_costs
df["OutputCost"] = output_costs 
df["TotalCost"] = total_costs

# SAVE RESULTS
df.to_csv("ai_teacher_evaluation_Operational_metrics.csv", index = False)

# DISPLAY DATAFRAME
from IPython.display import display

display(df[[
    "User Query",
    "GeneratedAnswer",
    "Latency",
    "InputTokens",
    "OutputTokens",
    "TotalTokens",
    "InputCost",
    "OutputCost",
    "TotalCost"
]])

Total question: 20


,User Query,GeneratedAnswer,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is AI?,"AI stands for Artificial Intelligence, which m...",22.460546,67,61,128,0.0,0.0,0.0
1,How does AI differ from traditional programming?,AI differs from traditional programming in tha...,11.169636,71,70,141,0.0,0.0,0.0
2,What is Machine Learning?,Machine Learning is a way for computers to lea...,11.056217,68,68,136,0.0,0.0,0.0
3,What is Deep Learning?,Deep learning is a type of artificial intellig...,9.370464,68,57,125,0.0,0.0,0.0
4,"What is the difference between AI, ML, and Dee...",Imagine a ladder of intelligence: AI (Artifici...,12.071299,76,77,153,0.0,0.0,0.0
5,What is training data?,Training data is the information used to teach...,8.507432,68,50,118,0.0,0.0,0.0
6,What is a Machine Learning model?,A Machine Learning model is a computer program...,10.338014,70,64,134,0.0,0.0,0.0
7,What is supervised learning?,Supervised learning is a type of machine learn...,9.676782,68,59,127,0.0,0.0,0.0
8,What is unsupervised learning?,Unsupervised learning is a type of machine lea...,11.493704,70,73,143,0.0,0.0,0.0
9,What is reinforcement learning?,Reinforcement learning is a way for machines t...,12.227159,68,79,147,0.0,0.0,0.0


In [2]:
# Load the NEW results
df = pd.read_csv("ai_teacher_evaluation_Operational_metrics.csv")

# CONVERT METRIC COLUMNS TO NUMERIC
metric_columns = [
    "Latency",
    "InputTokens",
    "OutputTokens",
    "TotalTokens",
    "InputCost",
    "OutputCost",
    "TotalCost"
]

for column in metric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Average metrics
average_latency = df["Latency"].mean()
average_input_tokens = df["InputTokens"].mean()
average_output_tokens = df["OutputTokens"].mean()
average_total_tokens = df["TotalTokens"].mean()
average_input_cost = df["InputCost"].mean()
average_output_cost = df["OutputCost"].mean()
average_total_cost = df["TotalCost"].mean()

# TOTAL METRICS

total_latency = df["Latency"].sum()
total_input_tokens = df["InputTokens"].sum()
total_output_tokens = df["OutputTokens"].sum()
total_tokens = df["TotalTokens"].sum()
total_input_cost = df["InputCost"].sum()
total_output_cost = df["OutputCost"].sum()
total_cost = df["TotalCost"].sum()

# CREATE OPERATIONAL METRICS DATAFRAME
operational_metrics_df = pd.DataFrame({
        "Metric": [
        "Latency (seconds)",
        "Input Tokens",
        "Output Tokens",
        "Total Tokens",
        "Input Cost (USD)",
        "Output Cost (USD)",
        "Total Cost (USD)"
    ],

    "Average": [
        average_latency,
        average_input_tokens,
        average_output_tokens,
        average_total_tokens,
        average_input_cost,
        average_output_cost,
        average_total_cost
    ],

    "Total": [
        total_latency,
        total_input_tokens,
        total_output_tokens,
        total_tokens,
        total_input_cost,
        total_output_cost,
        total_cost
    ]
})

# ROUND VALUES
operational_metrics_df["Average"] = (operational_metrics_df["Average"].round(6))
operational_metrics_df["Total"] = (operational_metrics_df["Total"].round(6))

# DISPLAY DATAFRAME
operational_metrics_df

,Metric,Average,Total
0,Latency (seconds),10.846902,216.938036
1,Input Tokens,69.050000,1381.000000
2,Output Tokens,61.850000,1237.000000
3,Total Tokens,130.900000,2618.000000
4,Input Cost (USD),0.000000,0.000000
5,Output Cost (USD),0.000000,0.000000
6,Total Cost (USD),0.000000,0.000000


In [3]:
operational_metrics_df.to_csv("ai_teacher_evaluation_Operational_metrics_average.csv", 
                              index=False, encoding="utf-8-sig")

In [6]:
# Versions
# pandas: 2.2.2
# requests: 2.34.2